# 3D görüntüleyici verisi

Tarayıcıdaki 3D simülasyon ekranı için yürüyüş kayıtlarını çıkarır: kemik konumları, kas yolları,
kas aktivasyonu ve kuvveti, ayak altı yer tepki kuvveti. Beş senaryo kaydedilir (normal, iki tekli
zayıflık, ikisi birlikte, ikisi birlikte + alışmış kontrolcü). **Eğitim yapılmaz, 10–15 dakika sürer.**

### Yapmanız gereken
1. **Çalışma zamanı → Çalışma zamanı türünü değiştir → CPU** seçip kaydedin.
2. **Çalışma zamanı → Tümünü çalıştır**, Drive izni istenince onay verin.
3. Sonunda `web_verisi.zip` bilgisayarınıza iner (İndirilenler klasörü). Dosyayı orada bırakın.

In [ ]:
# 1) Kurulum: Drive, kaynaklar, paketler
import hashlib, io, json, os, subprocess, sys, urllib.request, zipfile
from pathlib import Path, PurePosixPath
from google.colab import drive

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/kas_animasyon_colab')
(DRIVE / 'adaptation').mkdir(parents=True, exist_ok=True)
PROJECT = Path('/content/kas_animasyon')
(PROJECT / 'scripts').mkdir(parents=True, exist_ok=True)
(PROJECT / 'results').mkdir(exist_ok=True)
link = PROJECT / 'results' / 'adaptation'
if not link.exists():
    link.symlink_to(DRIVE / 'adaptation')

SCRIPTS = {"simulate_weakness_sweep.py": "\"\"\"Forward-simulation sweep: weaken muscle groups and let the published controller walk.\n\nEvery scenario starts from the same deterministic state as the baseline and\nuses the same controller without retraining. Only the active force component\n(gainprm[2]) of the chosen muscles is scaled; passive forces stay unchanged.\n\nThe environment's own termination (mean coordinate error above qpos_diff_th)\nis switched off so every run lasts three gait cycles unless the model falls.\nOutcomes are classified afterwards:\n  - walked:   mean coordinate error stayed under the controller threshold\n  - deviated: threshold crossed but the pelvis stayed up\n  - fell:     pelvis height dropped below FALL_RATIO of its initial value\n\"\"\"\nimport argparse\nfrom concurrent.futures import ProcessPoolExecutor\nfrom itertools import combinations\nimport json\nimport os\nfrom pathlib import Path\nimport sys\nimport time as clock\n\nimport numpy as np\n\nROOT = Path(__file__).resolve().parents[1]\nGROUPS = {\n    'right_tibialis_anterior': ['tibant_r'],\n    'right_soleus': ['soleus_r'],\n    'right_gastrocnemius': ['gasmed_r', 'gaslat_r'],\n    'right_vasti': ['vasint_r', 'vaslat_r', 'vasmed_r'],\n    'right_hamstrings': ['bflh_r', 'semimem_r', 'semiten_r'],\n    'right_gluteus_medius': ['glmed1_r', 'glmed2_r', 'glmed3_r'],\n    'right_gluteus_maximus': ['glmax1_r', 'glmax2_r', 'glmax3_r'],\n    'right_hip_flexors': ['iliacus_r', 'Ps_L1_VB_r', 'Ps_L1_TP_r', 'Ps_L1_L2_IVD_r', 'Ps_L2_TP_r',\n                          'Ps_L2_L3_IVD_r', 'Ps_L3_TP_r', 'Ps_L3_L4_IVD_r', 'Ps_L4_TP_r',\n                          'Ps_L4_L5_IVD_r', 'Ps_L5_TP_r', 'Ps_L5_VB_r'],\n}\nTHRESHOLD = 0.06   # controller's training threshold, rad (mean |qpos - ref|)\nFALL_RATIO = 0.7\n_worker = {}\n\n\ndef _constant_zero(_):\n    return 0.0\n\n\nINFERENCE_OBJECTS = {'buffer_size': 1, 'learning_rate': 0.0, 'lr_schedule': _constant_zero}\n\n\ndef init_worker(model_path=None, env_path=None):\n    os.environ.setdefault('OMP_NUM_THREADS', '1')\n    import torch\n    torch.set_num_threads(1)\n    sys.path.insert(0, str(ROOT / 'vendor/msgym/DynSyn-SAC/SB3-Scripts'))\n    import msgym  # noqa: F401  registers the environments\n    from DynSyn import SAC_DynSyn\n    checkpoint = ROOT / 'vendor/checkpoints/LocomotionLegs'\n    _worker['config'] = json.loads((checkpoint / 'locomotionLegs.json').read_text())\n    # Inference only: skip the ~2.8 GiB training replay buffer, and do not unpickle\n    # the saved learning-rate function (its bytecode is Python-version specific).\n    _worker['policy'] = SAC_DynSyn.load(str(model_path or checkpoint / 'checkpoint/best_model.zip'), device='cpu',\n                                        custom_objects=INFERENCE_OBJECTS)\n    _worker['env_path'] = str(env_path or checkpoint / 'checkpoint/best_env.zip')\n\n\ndef rollout(job):\n    import gymnasium as gym\n    import mujoco\n    from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize\n    from wrapper import MuscleNormWrapper\n\n    groups, capacity, seed, noise = job\n    config, policy = _worker['config'], _worker['policy']\n    options = dict(config['single_env_kwargs'])\n    options.update(random_init=False, reset_noise_scale=noise, gait_cycles=3, qpos_diff_th=1e9)\n    env = MuscleNormWrapper(gym.make(config['env_name'], render_mode=None, **options))\n    base = env.unwrapped\n    model, data = base.model, base.data\n    normalizer = VecNormalize.load(_worker['env_path'], DummyVecEnv([lambda: env]))\n    normalizer.training = False\n    normalizer.norm_reward = False\n    muscles = [m for g in groups for m in GROUPS[g]]\n    ids = [model.actuator(m).id for m in muscles]\n    passive = model.actuator_biasprm.copy()\n    if np.any(model.actuator_gainprm[ids, 2] <= 0):\n        raise ValueError('Expected explicit active force parameter')\n    model.actuator_gainprm[ids, 2] *= capacity\n    if not np.array_equal(passive, model.actuator_biasprm):\n        raise ValueError('Passive parameters changed')\n    if noise > 0:\n        normalizer.seed(seed)\n    observation = normalizer.reset()\n    pelvis = model.body('pelvis').id\n    start_height = float(data.xpos[pelvis, 2])\n    times, errors, heights = [], [], []\n    end_time = 3 * base.terminate_time\n    try:\n        while True:\n            action, _ = policy.predict(observation, deterministic=True)\n            observation, _, done, _ = normalizer.step(action)\n            if done[0]:\n                break\n            times.append(float(data.time))\n            errors.append(float(np.abs(data.qpos[3:] - base.qpos_ref[3:]).mean()))\n            heights.append(float(data.xpos[pelvis, 2]))\n            if heights[-1] < FALL_RATIO * start_height:\n                break\n    finally:\n        normalizer.close()\n    times, errors, heights = map(np.asarray, (times, errors, heights))\n    over = np.flatnonzero(errors > THRESHOLD)\n    fell = heights.size and heights.min() < FALL_RATIO * start_height\n    outcome = 'fell' if fell else ('deviated' if over.size else 'walked')\n    return {\n        'groups': list(groups), 'remaining_active_capacity': capacity, 'outcome': outcome,\n        'seed': seed, 'reset_noise_scale': noise,\n        'simulated_time_s': float(times[-1]) if times.size else 0.0,\n        'requested_time_s': float(end_time),\n        'first_threshold_crossing_s': float(times[over[0]]) if over.size else None,\n        'max_mean_coordinate_error_rad': float(errors.max()),\n        'mean_coordinate_error_rad': float(errors.mean()),\n        'min_pelvis_height_ratio': float(heights.min() / start_height),\n        'error_trace': errors[::5].round(5).tolist(),\n    }\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument('--levels', type=float, nargs='+', default=[0.6, 0.4, 0.2, 0.0])\n    parser.add_argument('--workers', type=int, default=max(1, (os.cpu_count() or 2) - 2))\n    parser.add_argument('--singles-only', action='store_true')\n    parser.add_argument('--output', default='results/forward_sweep')\n    parser.add_argument('--scenario', nargs='+', help='Groups joined by +; replaces the default grid')\n    parser.add_argument('--seeds', type=int, default=1, help='Repeats with reset noise (seed 0..n-1)')\n    parser.add_argument('--noise', type=float, default=0.0, help='Reset noise scale (training used 0.001)')\n    args = parser.parse_args()\n    scenarios = [(g,) for g in GROUPS]\n    if not args.singles_only:\n        scenarios += list(combinations(GROUPS, 2))\n    if args.scenario:\n        scenarios = [tuple(v.split('+')) for v in args.scenario]\n        if any(g not in GROUPS for s in scenarios for g in s):\n            raise ValueError('Unknown group')\n    seeds = range(args.seeds)\n    jobs = [((), 1.0, k, args.noise) for k in seeds] +            [(s, c, k, args.noise) for c in args.levels for s in scenarios for k in seeds]\n    output = ROOT / args.output\n    output.mkdir(parents=True, exist_ok=True)\n    results, started = [], clock.time()\n    with ProcessPoolExecutor(args.workers, initializer=init_worker) as pool:\n        for i, result in enumerate(pool.map(rollout, jobs), 1):\n            results.append(result)\n            print(f\"[{i}/{len(jobs)} {clock.time() - started:.0f}s] {result['remaining_active_capacity']:.2f} s{result['seed']} \"\n                  f\"{'+'.join(result['groups']) or 'baseline'}: {result['outcome']} \"\n                  f\"(max err {result['max_mean_coordinate_error_rad']:.3f}, t={result['simulated_time_s']:.2f})\",\n                  flush=True)\n            report = {'scope': 'Published LocomotionLegs controller, deterministic start, active force only, no retraining',\n                      'groups': GROUPS, 'threshold_rad': THRESHOLD, 'fall_ratio': FALL_RATIO,\n                      'interpretation': ('Outcomes describe this controller without adaptation. A failure does not '\n                                         'prove that a person or another controller could not walk.'),\n                      'results': results}\n            (output / 'sweep.json').write_text(json.dumps(report, indent=2), encoding='utf-8')\n\n\nif __name__ == '__main__':\n    main()\n", "export_web_data.py": "\"\"\"Export recorded walks for the browser viewer (web/).\n\nFor each scenario the published (or adapted) controller walks the model and\nevery control step (20 ms) is stored: bone poses, muscle paths resampled to a\nfixed number of points, activation, muscle force, vertical ground reaction\nforce under each foot, pelvis height and joint tracking error. Bone meshes are\nexported once. Everything goes to <out>/manifest.json, meshes.bin and one\n<scenario>.bin per scenario (little-endian float32 / uint32).\n\n  python scripts/export_web_data.py --adapted results/adaptation/glut_med_max_020 --out web/data\n\"\"\"\nimport argparse\nimport json\nfrom pathlib import Path\nimport sys\n\nimport numpy as np\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nimport simulate_weakness_sweep as sweep  # noqa: E402\n\nPOINTS = 8            # samples along each muscle path\nAFTER_FALL = 40       # control steps kept after a fall, for the animation\nBOTH = ('right_gluteus_medius', 'right_gluteus_maximus')\nSCENARIOS = [\n    # key, label, groups, capacity, controller, seed, noise\n    ('normal', 'Normal yürüyüş', (), 1.0, 'original', 0, 0.0),\n    ('glmed', 'Sadece gluteus medius %20', ('right_gluteus_medius',), 0.2, 'original', 0, 0.0),\n    ('glmax', 'Sadece gluteus maksimus %20', ('right_gluteus_maximus',), 0.2, 'original', 0, 0.0),\n    ('both', 'İkisi birlikte %20', BOTH, 0.2, 'original', 0, 0.0),\n    ('both_adapted', 'İkisi birlikte %20 + alışmış kontrolcü', BOTH, 0.2, 'adapted', 0, 0.001),\n]\nGROUP_LABELS = {\n    'tibialis_anterior': 'Tibialis anterior', 'soleus': 'Soleus', 'gastrocnemius': 'Gastroknemius',\n    'vasti': 'Vastuslar (kuadriseps)', 'hamstrings': 'Hamstring', 'gluteus_medius': 'Gluteus medius',\n    'gluteus_maximus': 'Gluteus maksimus', 'hip_flexors': 'Kalça fleksörleri (iliopsoas)',\n}\n\n\ndef resample(points, count):\n    if len(points) == 1:\n        return np.repeat(points, count, axis=0)\n    seg = np.linalg.norm(np.diff(points, axis=0), axis=1)\n    s = np.concatenate([[0], np.cumsum(seg)])\n    if s[-1] <= 0:\n        return np.repeat(points[:1], count, axis=0)\n    target = np.linspace(0, s[-1], count)\n    return np.stack([np.interp(target, s, points[:, k]) for k in range(3)], axis=1)\n\n\ndef load_policy(kind, adapted):\n    from DynSyn import SAC_DynSyn\n    checkpoint = ROOT / 'vendor/checkpoints/LocomotionLegs/checkpoint'\n    model_path, env_path = ((checkpoint / 'best_model.zip', checkpoint / 'best_env.zip') if kind == 'original'\n                            else (adapted / 'final_model.zip', adapted / 'final_env.zip'))\n    policy = SAC_DynSyn.load(str(model_path), device='cpu', custom_objects=sweep.INFERENCE_OBJECTS)\n    return policy, env_path\n\n\ndef record(scenario, adapted):\n    import gymnasium as gym\n    import mujoco\n    from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize\n    from wrapper import MuscleNormWrapper\n\n    key, label, groups, capacity, kind, seed, noise = scenario\n    config = sweep._worker['config']\n    policy, env_path = load_policy(kind, adapted)\n    options = dict(config['single_env_kwargs'])\n    options.update(random_init=False, reset_noise_scale=noise, gait_cycles=3, qpos_diff_th=1e9)\n    env = MuscleNormWrapper(gym.make(config['env_name'], render_mode=None, **options))\n    base = env.unwrapped\n    model, data = base.model, base.data\n    norm = VecNormalize.load(str(env_path), DummyVecEnv([lambda: env]))\n    norm.training = False\n    norm.norm_reward = False\n    full_force = model.actuator_gainprm[:, 2].copy()\n    ids = [model.actuator(m).id for g in groups for m in sweep.GROUPS[g]]\n    model.actuator_gainprm[ids, 2] *= capacity\n    if noise > 0:\n        norm.seed(seed)\n    obs = norm.reset()\n\n    geoms = [g for g in range(model.ngeom)\n             if model.geom_type[g] == mujoco.mjtGeom.mjGEOM_MESH and model.geom_group[g] < 3]\n    muscles = list(range(model.nu))\n    tendons = [int(model.actuator_trnid[a, 0]) for a in muscles]\n    act_index = model.actuator_actadr[muscles]\n    feet = {side: {model.body(f'calcn_{side}').id, model.body(f'toes_{side}').id} for side in 'rl'}\n    pelvis = model.body('pelvis').id\n    h0 = float(data.xpos[pelvis, 2])\n    frames = {k: [] for k in ('time', 'geom_pos', 'geom_quat', 'paths', 'act', 'force',\n                              'grf', 'foot', 'pelvis_h', 'error')}\n    quat = np.zeros(4)\n    force6 = np.zeros(6)\n\n    def keep():\n        frames['time'].append(data.time)\n        frames['geom_pos'].append(data.geom_xpos[geoms].copy())\n        qs = []\n        for g in geoms:\n            mujoco.mju_mat2Quat(quat, data.geom_xmat[g])\n            qs.append(quat.copy())\n        frames['geom_quat'].append(np.asarray(qs))\n        wrap_xpos = data.wrap_xpos.reshape(-1, 3)\n        wrap_obj = data.wrap_obj.reshape(-1)\n        paths = []\n        for t in tendons:\n            adr, num = int(data.ten_wrapadr[t]), int(data.ten_wrapnum[t])\n            pts = wrap_xpos[adr:adr + num][wrap_obj[adr:adr + num] != -2]\n            paths.append(resample(pts, POINTS) if len(pts) else np.zeros((POINTS, 3)))\n        frames['paths'].append(np.asarray(paths))\n        frames['act'].append(data.act[act_index].copy())\n        frames['force'].append(np.abs(data.actuator_force[muscles]))\n        grf = [0.0, 0.0]\n        for i in range(data.ncon):\n            c = data.contact[i]\n            bodies = {int(model.geom_bodyid[c.geom1]), int(model.geom_bodyid[c.geom2])}\n            for j, side in enumerate('rl'):\n                if bodies & feet[side]:\n                    mujoco.mj_contactForce(model, data, i, force6)\n                    grf[j] += abs(force6[0])\n        frames['grf'].append(grf)\n        frames['foot'].append([data.xpos[model.body(f'calcn_{s}').id].copy() for s in 'rl'])\n        frames['pelvis_h'].append(data.xpos[pelvis, 2])\n        frames['error'].append(np.abs(data.qpos[3:] - base.qpos_ref[3:]).mean())\n\n    keep()\n    fell_at, extra = None, None\n    while True:\n        obs, _, done, _ = norm.step(policy.predict(obs, deterministic=True)[0])\n        if done[0]:\n            break\n        keep()\n        if extra is not None:\n            extra -= 1\n            if extra == 0:\n                break\n        elif data.xpos[pelvis, 2] < sweep.FALL_RATIO * h0:\n            fell_at, extra = float(data.time), AFTER_FALL\n    norm.close()\n    arrays = {k: np.asarray(v, dtype=np.float32) for k, v in frames.items()}\n    errors = arrays['error'][: len(arrays['error']) if fell_at is None else\n                             int(np.searchsorted(arrays['time'], fell_at)) + 1]\n    outcome = 'fell' if fell_at is not None else ('deviated' if (errors > sweep.THRESHOLD).any() else 'walked')\n    info = {'key': key, 'label': label, 'groups': list(groups), 'capacity': capacity, 'controller': kind,\n            'seed': seed, 'reset_noise_scale': noise, 'outcome': outcome, 'fell_at_s': fell_at,\n            'frames': int(len(arrays['time']))}\n    return info, arrays, (model, geoms, muscles, full_force)\n\n\ndef export_meshes(model, geoms, out, reduction):\n    try:\n        import fast_simplification\n    except ImportError:\n        fast_simplification = None\n    used = sorted({int(model.geom_dataid[g]) for g in geoms})\n    blob, meshes = bytearray(), {}\n    for m in used:\n        v0, nv = int(model.mesh_vertadr[m]), int(model.mesh_vertnum[m])\n        f0, nf = int(model.mesh_faceadr[m]), int(model.mesh_facenum[m])\n        verts = model.mesh_vert[v0:v0 + nv].astype(np.float32)\n        faces = model.mesh_face[f0:f0 + nf].astype(np.int64)\n        if fast_simplification is not None and nf > 400 and reduction > 0:\n            verts, faces = fast_simplification.simplify(verts, faces, target_reduction=reduction)\n            verts, faces = verts.astype(np.float32), faces.astype(np.int64)\n        entry = {'vert_offset': len(blob), 'vert_count': int(len(verts))}\n        blob += verts.tobytes()\n        entry.update(face_offset=len(blob), face_count=int(len(faces)))\n        blob += faces.astype(np.uint32).tobytes()\n        meshes[m] = entry\n    (out / 'meshes.bin').write_bytes(bytes(blob))\n    order = {m: i for i, m in enumerate(used)}\n    return ([meshes[m] for m in used],\n            [{'mesh': order[int(model.geom_dataid[g])], 'body': model.body(int(model.geom_bodyid[g])).name,\n              'rgba': [round(float(x), 3) for x in model.geom_rgba[g]]} for g in geoms])\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument('--adapted', required=True, help='folder with final_model.zip / final_env.zip')\n    parser.add_argument('--out', default='web/data')\n    parser.add_argument('--eval-original', help='eval_original.json (summary counts)')\n    parser.add_argument('--eval-adapted', help='eval_final.json (summary counts)')\n    parser.add_argument('--mesh-reduction', type=float, default=0.6)\n    args = parser.parse_args()\n    out = Path(args.out)\n    out.mkdir(parents=True, exist_ok=True)\n    adapted = Path(args.adapted)\n    sweep.init_worker()\n\n    layout = ['time', 'geom_pos', 'geom_quat', 'paths', 'act', 'force', 'grf', 'foot', 'pelvis_h', 'error']\n    scenarios, model_info = [], None\n    for scenario in SCENARIOS:\n        info, arrays, model_info = record(scenario, adapted)\n        blob = bytearray()\n        info['arrays'] = {}\n        for name in layout:\n            a = np.ascontiguousarray(arrays[name], dtype=np.float32)\n            info['arrays'][name] = {'offset': len(blob), 'shape': list(a.shape)}\n            blob += a.tobytes()\n        (out / f\"{info['key']}.bin\").write_bytes(bytes(blob))\n        scenarios.append(info)\n        print(info['key'], info['outcome'], info['fell_at_s'], info['frames'], 'frames', flush=True)\n\n    model, geoms, muscles, full_force = model_info\n    meshes, geom_info = export_meshes(model, geoms, out, args.mesh_reduction)\n    group_of = {}\n    for g, names in sweep.GROUPS.items():\n        short = g.replace('right_', '')\n        for m in names:\n            group_of[m] = short\n            group_of[m[:-2] + '_l'] = short\n    muscle_info = []\n    for a in muscles:\n        name = model.actuator(a).name\n        muscle_info.append({'name': name, 'side': name[-1], 'group': group_of.get(name, 'other'),\n                            'max_force': round(float(full_force[a]), 1),\n                            'tau_act': round(float(model.actuator_dynprm[a, 0]), 4),\n                            'tau_deact': round(float(model.actuator_dynprm[a, 1]), 4)})\n    evals = {}\n    for kind, path in (('original', args.eval_original), ('adapted', args.eval_adapted)):\n        if path and Path(path).exists():\n            evals[kind] = json.loads(Path(path).read_text())['summary']\n    manifest = {'dt': float(model.opt.timestep) * 10, 'points_per_muscle': POINTS,\n                'threshold_rad': sweep.THRESHOLD, 'fall_ratio': sweep.FALL_RATIO,\n                'groups': {k: v for k, v in GROUP_LABELS.items()}, 'muscles': muscle_info,\n                'meshes': meshes, 'geoms': geom_info, 'scenarios': scenarios, 'layout': layout,\n                'evaluation': evals}\n    (out / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False), encoding='utf-8')\n    sizes = {p.name: round(p.stat().st_size / 1e6, 2) for p in out.iterdir()}\n    print(json.dumps(sizes, indent=1))\n\n\nif __name__ == '__main__':\n    main()\n"}
for name, text in SCRIPTS.items():
    (PROJECT / 'scripts' / name).write_text(text, encoding='utf-8')


def unpack(url, sha256, destination, strip_top=True):
    if destination.exists() and any(destination.iterdir()):
        return
    with urllib.request.urlopen(url, timeout=300) as response:
        payload = response.read()
    if hashlib.sha256(payload).hexdigest() != sha256:
        raise ValueError(f'Checksum mismatch: {url}')
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        for member in archive.infolist():
            parts = PurePosixPath(member.filename).parts[1 if strip_top else 0:]
            if not parts or member.is_dir():
                continue
            if any(p in ('.', '..') for p in parts):
                raise ValueError('Unsafe archive member')
            target = destination.joinpath(*parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(member))


SOURCES = [{"repository": "LNSGroup/msgym", "revision": "ad3aac166dab3577d7d16b4a03faccef94c7aac7", "archive_url": "https://codeload.github.com/LNSGroup/msgym/zip/ad3aac166dab3577d7d16b4a03faccef94c7aac7", "archive_sha256": "37dad71e1e79c3c53169ae4520134a9aedde8710afc4f4bf15a3491e8ee56c76", "archive_bytes": 20540062, "directory": "vendor\\msgym"}, {"repository": "LNSGroup/MS-Human-700", "revision": "2d686957aefd5739cf4d2859a6acfd94c8f84400", "archive_url": "https://codeload.github.com/LNSGroup/MS-Human-700/zip/2d686957aefd5739cf4d2859a6acfd94c8f84400", "archive_sha256": "65975d132606149e7707c8ce0b6015c33e2d6fd50d15f00e416f961eb47327bc", "archive_bytes": 30783005, "directory": "vendor\\MS-Human-700"}]
vendor = PROJECT / 'vendor'
unpack(SOURCES[0]['archive_url'], SOURCES[0]['archive_sha256'], vendor / 'msgym')
unpack(SOURCES[1]['archive_url'], SOURCES[1]['archive_sha256'], vendor / 'msgym' / 'msgym' / 'MS-Human-700')
unpack('https://github.com/LNSGroup/msgym/releases/download/Checkpoints/LocomotionLegs.zip',
       '8f442634f9e90cf8cd38de22f74fc0da0b9d842b726cbaf812161533e82313f4',
       vendor / 'checkpoints', strip_top=False)

subprocess.run('apt-get -qq update && apt-get -qq install -y libosmesa6 > /dev/null', shell=True, check=True)
# Colab'ın hazır numpy ve torch sürümleri sabit tutulur; çalışan defterin altında
# kütüphane değişirse oturum çökebilir.
import importlib.metadata as meta
keep = [f'{p}=={meta.version(p)}' for p in ('numpy', 'torch')]
# Colab'da hazır gelen TensorFlow/TensorBoard, JAX ve eski Gym kontrolcü kütüphanesi
# tarafından kendiliğinden yükleniyor ve torch ile aynı süreçte çökmeye yol açıyor.
# Bu projede kullanılmıyorlar; bu oturumdan kaldırılırlar.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '--quiet', 'tensorflow', 'tensorboard',
                'tf-keras', 'jax', 'jaxlib', 'gym'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'mujoco==3.17.0', 'gymnasium==1.2.3',
                'stable-baselines3==2.7.1', 'fast-simplification', *keep], check=True)
os.environ.pop('MUJOCO_GL', None)          # görüntü ayarı yalnızca video adımında açılır
os.environ['PYTHONFAULTHANDLER'] = '1'      # çökme olursa yerini yazdırır
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
paths = [str(vendor / 'msgym'), str(vendor / 'msgym' / 'DynSyn-SAC'),
         str(vendor / 'msgym' / 'DynSyn-SAC' / 'SB3-Scripts'), str(PROJECT / 'scripts')]
os.environ['PYTHONPATH'] = os.pathsep.join(paths + [os.environ.get('PYTHONPATH', '')])
sys.path[:0] = paths
os.chdir(PROJECT)
NAME = 'glut_med_max_020'
OUT = DRIVE / 'adaptation' / NAME
print('Python', sys.version.split()[0], '| işlemci çekirdeği:', os.cpu_count())
print('Kurulum tamam.')

In [ ]:
# 2) Kayıtları çıkar
EVAL_ORIGINAL = {"gluteus_medius+gluteus_maximus @0.20": {"walked": 0, "deviated": 1, "fell": 5}, "gluteus_medius @0.20": {"walked": 6, "deviated": 0, "fell": 0}, "gluteus_maximus @0.20": {"walked": 6, "deviated": 0, "fell": 0}, "normal @1.00": {"walked": 6, "deviated": 0, "fell": 0}}
Path('/content/eval_original.json').write_text(json.dumps({'summary': EVAL_ORIGINAL}), encoding='utf-8')
if not (OUT / 'final_model.zip').exists():
    raise RuntimeError('Drive\'da eğitilmiş kontrolcü bulunamadı (kas_animasyon_colab/adaptation/glut_med_max_020).')
export = subprocess.run([sys.executable, '-X', 'faulthandler', 'scripts/export_web_data.py',
                         '--adapted', str(OUT), '--out', '/content/web_data',
                         '--eval-original', '/content/eval_original.json',
                         '--eval-adapted', str(OUT / 'eval_final.json')],
                        capture_output=True, text=True)
print(export.stdout[-3000:])
if export.returncode != 0:
    lines = export.stderr.splitlines()
    print('\n'.join(l for l in lines if 'Extension modules' not in l)[-4000:])
    raise RuntimeError('Kayıt çıkarılamadı. Yukarıdaki metni paylaşın.')

In [ ]:
# 3) İndir (Drive'a da kopyalanır)
import shutil
from google.colab import files
archive = shutil.make_archive('/content/web_verisi', 'zip', root_dir='/content/web_data')
shutil.copy(archive, DRIVE / 'web_verisi.zip')
print('Boyut:', round(Path(archive).stat().st_size / 1e6, 1), 'MB')
files.download(archive)